# Calibration Toolbox - Basic Usage

This notebook demonstrates the basic usage of the Calibration Toolbox library for evaluating machine learning model calibration.

## Table of Contents
1. Installation and Setup
2. Computing Calibration Metrics
3. Visualizing Calibration
4. Comparing Models
5. Advanced Usage

---

## 1. Installation and Setup

In [ ]:
# Install the package (if not already installed)
# !pip install -e ..

import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')

# Import calibration toolbox
import calibration_toolbox as cal

print(f"Calibration Toolbox version: {cal.__version__}")

## 2. Generate Sample Data

Let's create some sample prediction data to demonstrate the metrics.

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate well-calibrated predictions
n_samples = 1000
n_classes = 3

# Create probabilities with some noise
true_probs = np.random.dirichlet(np.ones(n_classes), size=n_samples)
probabilities_calibrated = true_probs + np.random.normal(0, 0.05, (n_samples, n_classes))
probabilities_calibrated = np.clip(probabilities_calibrated, 0, 1)
probabilities_calibrated = probabilities_calibrated / probabilities_calibrated.sum(axis=1, keepdims=True)

# Generate labels based on probabilities
labels = np.array([np.random.choice(n_classes, p=prob) for prob in probabilities_calibrated])

print(f"Generated {n_samples} samples with {n_classes} classes")
print(f"Probabilities shape: {probabilities_calibrated.shape}")
print(f"Labels shape: {labels.shape}")

In [ ]:
# Create overconfident predictions (for comparison)
probabilities_overconfident = true_probs ** 0.5  # Sharpen probabilities
probabilities_overconfident = probabilities_overconfident / probabilities_overconfident.sum(axis=1, keepdims=True)

# Create underconfident predictions
probabilities_underconfident = true_probs ** 2  # Flatten probabilities
probabilities_underconfident = probabilities_underconfident / probabilities_underconfident.sum(axis=1, keepdims=True)

print("Created three sets of predictions:")
print("1. Calibrated (close to true probabilities)")
print("2. Overconfident (too confident)")
print("3. Underconfident (not confident enough)")

## 3. Computing Calibration Metrics

### Basic Metrics: ECE, MCE, RMSCE

In [ ]:
# Compute calibration metrics for each set of predictions
metrics_names = ['ECE', 'MCE', 'RMSCE']

print("Calibration Metrics Comparison:")
print("=" * 60)
print(f"{'Metric':<10} {'Calibrated':<15} {'Overconfident':<15} {'Underconfident':<15}")
print("=" * 60)

# Expected Calibration Error (ECE)
ece_cal = cal.ECE(probabilities_calibrated, labels)
ece_over = cal.ECE(probabilities_overconfident, labels)
ece_under = cal.ECE(probabilities_underconfident, labels)
print(f"{'ECE':<10} {ece_cal:<15.4f} {ece_over:<15.4f} {ece_under:<15.4f}")

# Maximum Calibration Error (MCE)
mce_cal = cal.MCE(probabilities_calibrated, labels)
mce_over = cal.MCE(probabilities_overconfident, labels)
mce_under = cal.MCE(probabilities_underconfident, labels)
print(f"{'MCE':<10} {mce_cal:<15.4f} {mce_over:<15.4f} {mce_under:<15.4f}")

# Root Mean Square Calibration Error (RMSCE)
rmsce_cal = cal.RMSCE(probabilities_calibrated, labels)
rmsce_over = cal.RMSCE(probabilities_overconfident, labels)
rmsce_under = cal.RMSCE(probabilities_underconfident, labels)
print(f"{'RMSCE':<10} {rmsce_cal:<15.4f} {rmsce_over:<15.4f} {rmsce_under:<15.4f}")

print("=" * 60)

### Class-Conditional Metrics: ACE, SCE

In [ ]:
# Adaptive Calibration Error (ACE)
ace_cal = cal.ACE(probabilities_calibrated, labels)
ace_over = cal.ACE(probabilities_overconfident, labels)
ace_under = cal.ACE(probabilities_underconfident, labels)

# Static Calibration Error (SCE)
sce_cal = cal.SCE(probabilities_calibrated, labels)
sce_over = cal.SCE(probabilities_overconfident, labels)
sce_under = cal.SCE(probabilities_underconfident, labels)

print("Class-Conditional Metrics:")
print("=" * 60)
print(f"{'Metric':<10} {'Calibrated':<15} {'Overconfident':<15} {'Underconfident':<15}")
print("=" * 60)
print(f"{'ACE':<10} {ace_cal:<15.4f} {ace_over:<15.4f} {ace_under:<15.4f}")
print(f"{'SCE':<10} {sce_cal:<15.4f} {sce_over:<15.4f} {sce_under:<15.4f}")
print("=" * 60)

## 4. Visualizing Calibration

### Reliability Diagrams

In [ ]:
# Create reliability diagrams for all three models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Calibrated model
cal.reliability_diagram(probabilities_calibrated, labels, 
                       title=f"Calibrated Model (ECE: {ece_cal:.4f})",
                       figsize=(6, 6), return_fig=False)
plt.subplot(1, 3, 1)

# Overconfident model  
cal.reliability_diagram(probabilities_overconfident, labels,
                       title=f"Overconfident Model (ECE: {ece_over:.4f})",
                       figsize=(6, 6), return_fig=False)
plt.subplot(1, 3, 2)

# Underconfident model
cal.reliability_diagram(probabilities_underconfident, labels,
                       title=f"Underconfident Model (ECE: {ece_under:.4f})",
                       figsize=(6, 6), return_fig=False)
plt.subplot(1, 3, 3)

plt.tight_layout()
plt.show()

### Confidence Histograms

In [ ]:
# Create confidence histograms
cal.confidence_histogram(probabilities_calibrated, labels,
                        title="Calibrated Model - Confidence Distribution")
plt.show()

cal.confidence_histogram(probabilities_overconfident, labels,
                        title="Overconfident Model - Confidence Distribution")
plt.show()

### Calibration Error Decomposition

In [ ]:
# Compare different metrics for the calibrated model
cal.calibration_error_decomposition(probabilities_calibrated, labels)
plt.show()

# Compare for the overconfident model
cal.calibration_error_decomposition(probabilities_overconfident, labels)
plt.show()

## 5. Advanced Usage

### Using the General Calibration Error Framework

In [ ]:
# The GCE function allows full customization
from calibration_toolbox import general_calibration_error

# ECE: L1 norm, not class-conditional
ece = general_calibration_error(
    probabilities_calibrated, labels,
    n_bins=15,
    class_conditional=False,
    norm=1
)

# MCE: L-infinity norm
mce = general_calibration_error(
    probabilities_calibrated, labels,
    n_bins=15,
    class_conditional=False,
    norm='inf'
)

# Custom: L3 norm, class-conditional, adaptive bins
custom_ce = general_calibration_error(
    probabilities_calibrated, labels,
    n_bins=10,
    class_conditional=True,
    adaptive_bins=True,
    norm=3
)

print(f"ECE (via GCE): {ece:.4f}")
print(f"MCE (via GCE): {mce:.4f}")
print(f"Custom CE (L3, class-cond, adaptive): {custom_ce:.4f}")

### Working with Logits

In [ ]:
# If your model outputs logits instead of probabilities
logits = np.random.randn(100, 3)
binary_labels = np.random.randint(0, 3, size=100)

# Set logits=True to apply softmax
ece_from_logits = cal.ECE(logits, binary_labels, logits=True)
print(f"ECE computed from logits: {ece_from_logits:.4f}")

### Class-wise Calibration Curves

In [ ]:
# Visualize per-class calibration
cal.class_wise_calibration_curve(probabilities_calibrated, labels,
                                title="Per-Class Calibration (Calibrated Model)")
plt.show()

cal.class_wise_calibration_curve(probabilities_overconfident, labels,
                                title="Per-Class Calibration (Overconfident Model)")
plt.show()

## Summary

In this notebook, we've demonstrated:

1. **Basic Metrics**: ECE, MCE, RMSCE for measuring overall calibration
2. **Class-Conditional Metrics**: ACE and SCE for per-class calibration analysis
3. **Visualizations**: Reliability diagrams, confidence histograms, and metric comparisons
4. **Advanced Usage**: The flexible GCE framework for custom calibration metrics
5. **Practical Tips**: Working with logits and analyzing class-wise calibration

### Key Takeaways

- **Lower calibration error** = Better calibration (values closer to 0)
- **ECE** is the most commonly used metric (L1 norm, top-1 prediction)
- **MCE** captures the worst-case bin error (useful for safety-critical applications)
- **ACE/SCE** provide class-conditional analysis for multi-class problems
- **Reliability diagrams** should show points close to the diagonal for well-calibrated models

### Next Steps

- Explore the [documentation](../docs/index.rst) for detailed API reference
- Check out research papers in [papers.md](../papers.md) for theoretical background
- Run the test suite to understand edge cases and validation

---

**Calibration Toolbox** - A lightweight, framework-agnostic library for model calibration evaluation.